In [ ]:
_repo_root = !git rev-parse --show-toplevel
%cd {_repo_root[0]}
del _repo_root

# Bivariate fitting

Thin wrapper around `paper.experiments.bivariate`.


In [ ]:
import matplotlib as mpl
import pandas as pd

from paper.experiments.bivariate import DEFAULT_RUNS_DIR, PROFILES, run_profile
from paper.plotting import plot_bivariate_target_gallery, plot_scaling
from paper.targets import TargetSpec
from paper.tuning import summarize_raw

mpl.rcParams["figure.dpi"] = 140


In [ ]:
profile_name = "small"
profile = PROFILES[profile_name]
raw_path = DEFAULT_RUNS_DIR / f"bivariate_{profile_name}.csv"
summary_path = DEFAULT_RUNS_DIR / f"bivariate_{profile_name}_summary.csv"
plot_target_kind = "monotone"

In [ ]:
if raw_path.exists():
    raw = pd.read_csv(raw_path)
else:
    raw = run_profile(profile)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(raw_path, index=False)

expected_models = {"unconstrained", "monotone"}
if not expected_models <= set(raw["model"].unique()):
    raise ValueError(f"{raw_path} does not contain {sorted(expected_models)}; rerun this profile")

if set(raw["complexity"]) != set(profile.complexities):
    raise ValueError(f"{raw_path} does not match {profile_name!r} complexities; rerun this profile")

summary = summarize_raw(raw, profile.budgets)
summary.to_csv(summary_path, index=False)
plot_summary = summary.loc[summary["target_kind"] == plot_target_kind].copy()
plot_summary.head()


In [ ]:
plot_scaling(plot_summary);

In [ ]:
plot_bivariate_target_gallery([
    TargetSpec(kind=plot_target_kind, complexity=complexity, seed=0)
    for complexity in profile.complexities
]);